# HASTIKA @ ICON-2026 — Task A: Binary Hate Speech Detection
### Kannada-English Code-Mixed Comments — Hate (1) vs Non-Hate (0)

This notebook implements the full pipeline for **Task A only**:

```
Inspect Dataset -> Identify Columns -> Clean/Normalize -> Train/Val/Test Split
      -> [TF-IDF + Logistic Regression]  AND  [MuRIL / XLM-R Fine-tuning]
      -> Evaluation (F1, Precision, Recall) -> Error Analysis
      -> Pick Best Model -> Competition Submission
```

**Data used (from the official repo https://github.com/shankarb14/Hastika-ICON2026):**
- `data/binary_train.csv` — 6,446 labeled rows, columns `id, Comment, Label` (`Label` is `Hate`/`Non-Hate`)
- `data/binary_validation_inputs.csv` — 806 unlabeled rows, columns `id, Comment` (what you predict on right now)
- You only need these two CSVs — not the `.py`/`.md` files in the repo.

**How to use this notebook on Kaggle (read this first):**
1. Download just those two CSVs from the `data/` folder of the repo, then Kaggle -> **Datasets -> New Dataset** -> upload them -> **Create**.
2. Open this notebook, and in the right sidebar under **Add Input**, attach the dataset you just created.
3. Under **Settings -> Accelerator**, select **GPU T4 x2** (or any available GPU). The transformer fine-tuning cell needs a GPU.
4. Run the cells **top to bottom, in order**. Do not skip cells.
5. In the **CONFIG cell** below, fix the folder name in `TRAIN_PATH`/`TEST_PATH` to match your uploaded dataset's actual folder name on Kaggle (run the "Inspect Dataset" cell if unsure).
6. At the end, `predictions.zip` (containing `predictions.csv`) is created in `/kaggle/working/` — download it from the **Output** tab and upload it to CodaBench: https://www.codabench.org/competitions/17784/


## Step 0 — Install & Import Libraries

In [ ]:
# Run this once. Versions are pinned to avoid known conflicts on Kaggle's preinstalled environment
# (pandas 3.x / numpy 2.1+ break several preinstalled packages, and older transformers lacks some TrainingArguments options).
!pip install -q -U "pandas<3.0" "numpy<2.1" "transformers>=4.44" datasets accelerate evaluate scikit-learn matplotlib seaborn emoji


**After this cell finishes, restart the kernel once** (Kaggle menu -> Run -> Restart & Clear Cell Outputs), then run all cells from the top again. This makes sure the pinned versions actually load instead of the ones Kaggle had preloaded into memory.

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix
)

import torch
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

import emoji
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU, then Restart & Run All.")


## Step 1 — CONFIG (EDIT THIS CELL)

Set your dataset paths. On Kaggle, an attached dataset lives under `/kaggle/input/<dataset-folder-name>/`.
Run the "Inspect Dataset" cell first if you are not sure of the exact filenames — it will list everything under `/kaggle/input/`.

- `TRAIN_PATH`: the labeled training file (required).
- `TEST_PATH`: the official unlabeled test file used for the competition submission. If you don't have one yet, set it to `None` and the notebook will carve a held-out test split from `TRAIN_PATH` instead, purely so you can see evaluation numbers.
- `TEXT_COL`, `LABEL_COL`, `ID_COL`: leave as `None` to let the notebook auto-detect them. If auto-detection picks the wrong column (check the printed output in Step 2), type the exact column names here manually.
- `MODEL_NAME`: choose `"google/muril-base-cased"` (built for Indian languages incl. Kannada, recommended) or `"xlm-roberta-base"` (strong multilingual baseline).


In [ ]:
# ==================== EDIT THESE VALUES ====================
# These paths/column names match the official HASTIKA repo (data/binary_train.csv, data/binary_validation_inputs.csv).
# Update the dataset folder name below to whatever you named it when you uploaded it to Kaggle.
TRAIN_PATH = "/kaggle/input/hastika-dataset/binary_train.csv"              # columns: id, Comment, Label
TEST_PATH  = "/kaggle/input/hastika-dataset/binary_validation_inputs.csv"  # columns: id, Comment (NO labels)

TEXT_COL  = "Comment"   # official column name -- change only if your file differs
LABEL_COL = "Label"     # official column name -- change only if your file differs
ID_COL    = "id"        # official column name -- change only if your file differs

MODEL_NAME = "google/muril-base-cased"   # or "xlm-roberta-base"
OUTPUT_DIR = "/kaggle/working/"
MAX_LEN = 128
# =============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


## Step 2 — Inspect Dataset

In [ ]:
# List everything Kaggle has attached, in case your path above is wrong
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
print("Shape:", train_df.shape)
print("\nColumns:", list(train_df.columns))
print("\nDtypes:\n", train_df.dtypes)
train_df.head(10)


In [ ]:
print(train_df.isnull().sum())


## Step 3 — Identify Text + Hate Label Columns (auto-detect with manual override)

In [ ]:
def auto_detect_column(df, candidates, exclude=()):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        for lower, original in cols_lower.items():
            if cand in lower and original not in exclude:
                return original
    return None

if TEXT_COL is None:
    TEXT_COL = auto_detect_column(train_df, ["comment", "text", "tweet", "sentence", "content", "sample"])
if LABEL_COL is None:
    LABEL_COL = auto_detect_column(train_df, ["hate", "label", "class", "target"], exclude=(TEXT_COL,))
if ID_COL is None:
    ID_COL = auto_detect_column(train_df, ["id"], exclude=(TEXT_COL, LABEL_COL))

print("Detected TEXT_COL  :", TEXT_COL)
print("Detected LABEL_COL :", LABEL_COL)
print("Detected ID_COL    :", ID_COL)

assert TEXT_COL is not None, "Could not auto-detect the text column. Set TEXT_COL manually in the CONFIG cell."
assert LABEL_COL is not None, "Could not auto-detect the label column. Set LABEL_COL manually in the CONFIG cell."

print("\nLabel value counts:")
print(train_df[LABEL_COL].value_counts(dropna=False))


In [ ]:
# Normalize label values to integers 0/1.
# Handles cases where labels are given as strings like 'Hate'/'Non-Hate', 'HOF'/'NOT', 'yes'/'no', or already 0/1.
def normalize_label(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    hate_tokens = {"1", "hate", "hof", "yes", "true", "offensive", "hateful"}
    nonhate_tokens = {"0", "non-hate", "nonhate", "not", "no", "false", "not-offensive", "non_hate"}
    if s in hate_tokens:
        return 1
    if s in nonhate_tokens:
        return 0
    try:
        return int(float(s))
    except ValueError:
        raise ValueError(f"Unrecognized label value: {val!r}. Add it to normalize_label().")

train_df["label_int"] = train_df[LABEL_COL].apply(normalize_label)
train_df = train_df.dropna(subset=["label_int", TEXT_COL]).reset_index(drop=True)
train_df["label_int"] = train_df["label_int"].astype(int)
print(train_df["label_int"].value_counts())


## Step 4 — Data Cleaning / Normalization

Code-mixed Kannada-English text needs care: we keep both scripts intact, strip noise (URLs, mentions, HTML), and normalize punctuation/whitespace without destroying meaningful code-switching.

**Emojis and text-emoticons are handled deliberately, not just deleted.** A raw regex that strips all non-alphanumeric characters would silently destroy emojis (😡🤬❤️) and emoticons (`:)`, `xD`, `:(`) — but these often carry the actual sentiment of a comment and can flip how a sentence should be read (e.g. an insult followed by `xD` usually signals mocking/sarcasm rather than a straightforwardly hateful tone). So before any stripping happens, we:
1. **Detect** every emoji and emoticon in the raw dataset (for inspection/analysis, exported to CSV).
2. **Convert** each one into a descriptive word (e.g. 😡 -> `angry_face`, `xD` -> `laughing_face`) so the model can actually learn from that signal as normal text, instead of losing it or choking on an unfamiliar glyph.


In [ ]:
# ---- 4a. Detect emojis (for inspection) ----
def extract_emojis(text):
    # emoji.EMOJI_DATA also contains a few non-emoji symbols like (c)/(r)/(tm) -- exclude those
    return "".join(
        c for c in str(text)
        if c in emoji.EMOJI_DATA and c not in ("\u00a9", "\u00ae", "\u2122")  # (c) (r) (tm)
    )

def count_emojis(text):
    return len(extract_emojis(text))

train_df["emojis_found"] = train_df[TEXT_COL].apply(extract_emojis)
train_df["emoji_count"] = train_df["emojis_found"].apply(len)

print(f"Comments containing at least one real emoji: {(train_df['emoji_count'] > 0).sum()} / {len(train_df)}")
print("\nMost common emojis in the dataset:")
all_emojis = "".join(train_df["emojis_found"])
print(Counter(all_emojis).most_common(20))
print("\nAverage emoji count by label (label_int: 0=Non-Hate, 1=Hate):")
print(train_df.groupby("label_int")["emoji_count"].mean())


In [ ]:
# ---- 4b. Detect text-based emoticons (for inspection) ----
EMOTICON_PATTERN = re.compile(
    r"(?::|;|=|[xX])(?:-)?(?:\)|\(|[dD]|[pP]|[oO]|\||\\|/|\*)"
    r"|<3|:'\(|:'\)|:-\)|:-\("
)

def extract_emoticons(text):
    return EMOTICON_PATTERN.findall(str(text))

train_df["emoticons_found"] = train_df[TEXT_COL].apply(extract_emoticons)
train_df["emoticon_count"] = train_df["emoticons_found"].apply(len)

print(f"Comments containing a text emoticon: {(train_df['emoticon_count'] > 0).sum()} / {len(train_df)}")
all_emoticons = [e for row in train_df["emoticons_found"] for e in row]
print(Counter(all_emoticons).most_common(20))
print("\nAverage emoticon count by label:")
print(train_df.groupby("label_int")["emoticon_count"].mean())


In [ ]:
# ---- 4c. Export a CSV so you can visually inspect exactly which rows have emojis/emoticons ----
inspection_cols = [ID_COL, TEXT_COL, "emojis_found", "emoji_count", "emoticons_found", "emoticon_count", "label_int"]
inspection_cols = [c for c in inspection_cols if c is not None and c in train_df.columns]
train_df[inspection_cols].to_csv(os.path.join(OUTPUT_DIR, "emoji_emoticon_inspection.csv"), index=False)
print("Saved:", os.path.join(OUTPUT_DIR, "emoji_emoticon_inspection.csv"))
train_df[inspection_cols].head(10)


In [ ]:
# ---- 4d. Cleaning function: convert emojis/emoticons to words BEFORE stripping symbols, so meaning is preserved ----
KANNADA_RANGE = r"\u0C80-\u0CFF"

EMOTICON_MAP = {
    r"(?::|;|=)(?:-)?\)": " happy_face ",
    r"(?::|;|=)(?:-)?\(": " sad_face ",
    r"(?::|;|=)(?:-)?[dD]": " laughing_face ",
    r"(?::|;|=)(?:-)?[pP]": " playful_face ",
    r"[xX][dD]": " laughing_face ",
    r":'\(": " crying_face ",
    r":'\)": " happy_tears_face ",
    r"<3": " heart ",
}

def normalize_emoticons(text):
    for pattern, replacement in EMOTICON_MAP.items():
        text = re.sub(pattern, replacement, text)
    return text

def clean_text(text):
    text = str(text)
    text = text.strip()
    # remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # remove @mentions
    text = re.sub(r"@\w+", " ", text)
    # remove HTML entities/tags
    text = re.sub(r"&\w+;", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    # convert real emojis to descriptive words BEFORE the character filter would delete them
    text = emoji.demojize(text, delimiters=(" ", " "))   # e.g. "😡" -> " angry_face "
    text = text.replace("_", " ")                         # "angry_face" -> "angry face" (cleaner tokens)
    # convert text emoticons (xD, :), :( etc.) to descriptive words
    text = normalize_emoticons(text)
    # keep Kannada script, English letters, digits, basic punctuation; drop any remaining stray symbols
    text = re.sub(rf"[^{KANNADA_RANGE}a-zA-Z0-9\s.,!?']", " ", text)
    # collapse repeated punctuation (e.g. "!!!!" -> "!")
    text = re.sub(r"([.,!?])\1+", r"\1", text)
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df[TEXT_COL].apply(clean_text)

# drop rows that became empty after cleaning
before = len(train_df)
train_df = train_df[train_df["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"Dropped {before - len(train_df)} empty rows after cleaning.")

train_df[[TEXT_COL, "clean_text", "label_int"]].head(10)


## Step 5 — Train / Validation / Test Split\n\nStratified split so the hate/non-hate ratio is preserved in every subset.

In [ ]:
HAS_OFFICIAL_TEST = TEST_PATH is not None and os.path.exists(TEST_PATH)

if HAS_OFFICIAL_TEST:
    # Official unlabeled test set exists -> just split train into train/val (85/15)
    train_split, val_split = train_test_split(
        train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_int"]
    )
    print("Using official test file for final submission.")
else:
    # No official test file -> carve train/val/test out of train_df (70/15/15)
    train_split, temp_split = train_test_split(
        train_df, test_size=0.30, random_state=SEED, stratify=train_df["label_int"]
    )
    val_split, test_split = train_test_split(
        temp_split, test_size=0.50, random_state=SEED, stratify=temp_split["label_int"]
    )
    print("No official test file found -> created a held-out internal test split for evaluation only.")

print("Train:", train_split.shape)
print("Val  :", val_split.shape)
if not HAS_OFFICIAL_TEST:
    print("Test :", test_split.shape)


---\n## Branch A — Traditional ML Baseline: TF-IDF + Logistic Regression

In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(train_split["clean_text"])
X_val_tfidf   = tfidf.transform(val_split["clean_text"])

y_train = train_split["label_int"].values
y_val   = val_split["label_int"].values

lr_clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    C=1.0,
    random_state=SEED
)
lr_clf.fit(X_train_tfidf, y_train)

val_preds_lr = lr_clf.predict(X_val_tfidf)

print("=== TF-IDF + Logistic Regression — Validation Results ===")
print(classification_report(y_val, val_preds_lr, target_names=["Non-Hate", "Hate"], digits=4))


---\n## Branch B — Transformer Fine-tuning (MuRIL / XLM-R)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HateDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

train_dataset = HateDataset(train_split["clean_text"], train_split["label_int"], tokenizer)
val_dataset   = HateDataset(val_split["clean_text"],   val_split["label_int"],   tokenizer)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

# Class weights to counter label imbalance -- computed from the ACTUAL training split.
# (The load-time "UNEXPECTED"/"MISSING" keys warning you may see above is normal: MuRIL/XLM-R's
# original pretraining head is discarded and a fresh classification head is added for this task.)
class_counts = train_split["label_int"].value_counts().sort_index().values  # [count_class0, count_class1]
class_weights_np = (1.0 / class_counts)
class_weights_np = class_weights_np / class_weights_np.sum() * len(class_counts)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(DEVICE)
print("Class counts (0=Non-Hate, 1=Hate):", class_counts)
print("Class weights used in loss:", class_weights_np)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="binary", zero_division=0),
        "recall": recall_score(labels, preds, average="binary", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "transformer_ckpt"),
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


In [ ]:
transformer_eval = trainer.evaluate()
print("=== Transformer (%s) — Validation Results ===" % MODEL_NAME)
for k, v in transformer_eval.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

val_preds_logits = trainer.predict(val_dataset).predictions
val_preds_transformer = np.argmax(val_preds_logits, axis=-1)

print()
print(classification_report(y_val, val_preds_transformer, target_names=["Non-Hate", "Hate"], digits=4))


## Step 6 — Evaluation: F1-score, Precision, Recall (side-by-side comparison)

In [ ]:
def summarize(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

results = pd.DataFrame([
    summarize("TF-IDF + Logistic Regression", y_val, val_preds_lr),
    summarize(f"Transformer ({MODEL_NAME})", y_val, val_preds_transformer),
])
results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, preds) in zip(
    axes,
    [("TF-IDF + LR", val_preds_lr), (f"Transformer", val_preds_transformer)]
):
    cm = confusion_matrix(y_val, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-Hate", "Hate"], yticklabels=["Non-Hate", "Hate"], ax=ax)
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()


## Step 7 — Error Analysis\n\nInspect the comments the transformer model gets wrong — this tells you what kinds of hate speech (sarcasm, implicit hate, code-switch-heavy text, etc.) the model struggles with.

In [ ]:
val_analysis = val_split.copy().reset_index(drop=True)
val_analysis["true_label"] = y_val
val_analysis["pred_label"] = val_preds_transformer
val_analysis["correct"] = val_analysis["true_label"] == val_analysis["pred_label"]

errors = val_analysis[~val_analysis["correct"]]
print(f"Total validation examples: {len(val_analysis)}")
print(f"Misclassified: {len(errors)} ({len(errors)/len(val_analysis)*100:.1f}%)")

false_positives = errors[(errors["true_label"] == 0) & (errors["pred_label"] == 1)]
false_negatives = errors[(errors["true_label"] == 1) & (errors["pred_label"] == 0)]

print(f"\nFalse Positives (predicted Hate, actually Non-Hate): {len(false_positives)}")
print(f"False Negatives (predicted Non-Hate, actually Hate): {len(false_negatives)}")

print("\n--- Sample False Positives ---")
display(false_positives[[TEXT_COL, "clean_text", "true_label", "pred_label"]].head(10))

print("\n--- Sample False Negatives ---")
display(false_negatives[[TEXT_COL, "clean_text", "true_label", "pred_label"]].head(10))


## Step 8 — Select Final Best Model\n\nPicks whichever model scored higher on macro-F1 (the standard metric for imbalanced hate-speech shared tasks). Edit the logic below if the organizers specify a different official metric.

In [ ]:
f1_lr = results.loc[results["model"] == "TF-IDF + Logistic Regression", "f1_macro"].values[0]
f1_tf = results.loc[results["model"].str.startswith("Transformer"), "f1_macro"].values[0]

if f1_tf >= f1_lr:
    BEST_MODEL = "transformer"
    print(f"Best model: Transformer ({MODEL_NAME}) | macro-F1 = {f1_tf:.4f} vs LR macro-F1 = {f1_lr:.4f}")
    trainer.save_model(os.path.join(OUTPUT_DIR, "best_model"))
    tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))
else:
    BEST_MODEL = "tfidf_lr"
    print(f"Best model: TF-IDF + Logistic Regression | macro-F1 = {f1_lr:.4f} vs Transformer macro-F1 = {f1_tf:.4f}")
    import joblib
    joblib.dump(lr_clf, os.path.join(OUTPUT_DIR, "best_model_lr.joblib"))
    joblib.dump(tfidf, os.path.join(OUTPUT_DIR, "best_model_tfidf.joblib"))


## Step 9 — Competition Submission\n\nRuns the best model on the official test file (or the held-out internal test split, if no official test file was provided) and writes `submission.csv`.

In [ ]:
if HAS_OFFICIAL_TEST:
    test_df = pd.read_csv(TEST_PATH)
else:
    test_df = test_split.copy()

# apply the SAME cleaning used for training
test_df["clean_text"] = test_df[TEXT_COL].apply(clean_text)

if ID_COL is not None and ID_COL in test_df.columns:
    submission_ids = test_df[ID_COL].values
else:
    submission_ids = np.arange(len(test_df))

if BEST_MODEL == "transformer":
    test_texts = test_df["clean_text"].tolist()
    test_encodings = tokenizer(
        test_texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt"
    )
    model.eval()
    all_preds = []
    batch_size = 32
    with torch.no_grad():
        for i in range(0, len(test_texts), batch_size):
            batch = {k: v[i:i+batch_size].to(DEVICE) for k, v in test_encodings.items()}
            logits = model(**batch).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
    final_preds = all_preds
else:
    X_test_tfidf = tfidf.transform(test_df["clean_text"])
    final_preds = lr_clf.predict(X_test_tfidf)

submission = pd.DataFrame({
    "id": submission_ids,
    "label": final_preds
})
submission["label"] = submission["label"].map({0: "Non-Hate", 1: "Hate"})  # matches official format: id,label with Hate/Non-Hate

# Official HASTIKA submission rules: file must be named exactly "predictions.csv",
# header "id,label", zipped ALONE (no enclosing folder), UTF-8 encoding.
predictions_path = os.path.join(OUTPUT_DIR, "predictions.csv")
submission.to_csv(predictions_path, index=False, encoding="utf-8")
print(f"Saved predictions to: {predictions_path}")
submission.head(10)


In [ ]:
import zipfile

zip_path = os.path.join(OUTPUT_DIR, "predictions.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(predictions_path, arcname="predictions.csv")  # arcname ensures no folder nesting inside the zip

print(f"Zipped submission ready at: {zip_path}")
print("Download this .zip from the Kaggle 'Output' tab and upload it to CodaBench:")
print("https://www.codabench.org/competitions/17784/")


## Notes / things to double-check before you submit

1. **Submission is on CodaBench, not Kaggle**: https://www.codabench.org/competitions/17784/ — download `predictions.zip` from Kaggle's **Output** tab (right sidebar after the notebook finishes) and upload it there.
2. **The real, gold-labeled test set isn't out yet.** Per the HASTIKA repo, test inputs release 20 Sep and the deadline is 01 Oct. Until then, `binary_validation_inputs.csv` (806 unlabeled rows) is what you predict on — your CodaBench leaderboard score right now reflects that validation phase, not the final test.
3. **If your uploaded Kaggle dataset folder has a different name**, update the folder segment in `TRAIN_PATH`/`TEST_PATH` in the CONFIG cell (e.g. `/kaggle/input/<your-folder-name>/binary_train.csv`).
4. **If you get an out-of-memory error** on the transformer cell, lower `per_device_train_batch_size` to 8 in `TrainingArguments`.
5. **To try both MuRIL and XLM-R**, just change `MODEL_NAME` in the CONFIG cell and `Restart & Run All` again — then compare the two `results` tables.
6. This notebook only implements **Task A**. Task B (fine-grained target category classification into Gender/Political/Religion/Geo-political/Violence/Others, using `data/multiclass_train.csv`) is a separate multi-class notebook — ask if you want that built next.
